![GovSpace Logo](https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg)

# 🔀 Multi-Step Agent Workflow with LangGraph

> **💡 Tip:** Create a folder with your name (e.g. `matt-grasser/`) and copy this notebook into it before editing. The `TEMPLATES` folder syncs from GitHub and is read-only.

In this notebook you'll build a **stateful agent workflow** using LangGraph — where multiple steps (nodes) are connected by a graph, and the agent can branch, loop, and make decisions.

## What you'll learn
1. How LangGraph models workflows as state machines
2. Building a research-then-summarize pipeline
3. Conditional routing between nodes
4. How state flows through the graph

In [ ]:
import os
from typing import Annotated, TypedDict
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

## Concept: Graphs as Workflows

LangGraph lets you model agent workflows as **directed graphs**:

```
[Research] → [Evaluate] → [Summarize] → END
                 ↓
            [Research]  (loop back if more research needed)
```

Each **node** is a function that transforms state. **Edges** define the flow.

## Step 1: Define State

State is a typed dictionary that flows through every node in the graph.

In [ ]:
class ResearchState(TypedDict):
    """State that flows through our research workflow."""
    topic: str
    messages: Annotated[list, add_messages]
    research_notes: list[str]
    research_rounds: int
    summary: str

## Step 2: Define Nodes

Each node is a function that takes state and returns updated state.

In [ ]:
model = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)


def research_node(state: ResearchState) -> dict:
    """Generate research notes on the topic."""
    round_num = state.get("research_rounds", 0) + 1
    existing_notes = state.get("research_notes", [])

    if existing_notes:
        context = "\n".join(f"- {n}" for n in existing_notes)
        prompt = (
            f"You are researching: {state['topic']}\n\n"
            f"Previous findings:\n{context}\n\n"
            f"Find 2-3 NEW angles or details not yet covered. "
            f"Be specific and factual. Return only bullet points."
        )
    else:
        prompt = (
            f"You are researching: {state['topic']}\n\n"
            f"Provide 3-4 key findings as bullet points. "
            f"Be specific and factual."
        )

    response = model.invoke([HumanMessage(content=prompt)])
    new_notes = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip().startswith("-")]

    print(f"📚 Research round {round_num}: found {len(new_notes)} points")
    for note in new_notes:
        print(f"   • {note[:80]}..." if len(note) > 80 else f"   • {note}")

    return {
        "research_notes": existing_notes + new_notes,
        "research_rounds": round_num,
        "messages": [response],
    }


def evaluate_node(state: ResearchState) -> dict:
    """Decide if we have enough research or need another round."""
    notes = state.get("research_notes", [])
    rounds = state.get("research_rounds", 0)

    print(f"\n🔍 Evaluating: {len(notes)} notes after {rounds} round(s)")
    # Simple heuristic: do 2 rounds, then move to summary
    return state


def summarize_node(state: ResearchState) -> dict:
    """Synthesize research notes into a coherent summary."""
    notes = state.get("research_notes", [])
    context = "\n".join(f"- {n}" for n in notes)

    prompt = (
        f"You researched: {state['topic']}\n\n"
        f"Research notes:\n{context}\n\n"
        f"Write a concise 2-3 paragraph summary synthesizing these findings. "
        f"Highlight the most important insights."
    )

    response = model.invoke([HumanMessage(content=prompt)])
    print(f"\n📝 Summary generated ({len(response.content)} chars)")

    return {
        "summary": response.content,
        "messages": [response],
    }

## Step 3: Define Routing Logic

Conditional edges let the graph decide where to go next based on state.

In [ ]:
def should_continue_research(state: ResearchState) -> str:
    """Decide: do more research or move to summary?"""
    rounds = state.get("research_rounds", 0)
    if rounds < 2:
        print("   → More research needed")
        return "research"  # loop back
    else:
        print("   → Enough research, moving to summary")
        return "summarize"  # move forward

## Step 4: Build the Graph

In [ ]:
# Create the graph
workflow = StateGraph(ResearchState)

# Add nodes
workflow.add_node("research", research_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("summarize", summarize_node)

# Define edges
workflow.set_entry_point("research")
workflow.add_edge("research", "evaluate")
workflow.add_conditional_edges(
    "evaluate",
    should_continue_research,
    {"research": "research", "summarize": "summarize"},
)
workflow.add_edge("summarize", END)

# Compile
app = workflow.compile()

print("Graph compiled! Nodes:", list(app.get_graph().nodes.keys()))

## Step 5: Run the Workflow

In [ ]:
result = app.invoke({
    "topic": "How AI agents are being used in government technology (govtech)",
    "messages": [],
    "research_notes": [],
    "research_rounds": 0,
    "summary": "",
})

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(result["summary"])

## 🧪 Try It Yourself

Change the topic and run again:

In [ ]:
# Try your own topic!
result = app.invoke({
    "topic": "YOUR TOPIC HERE",
    "messages": [],
    "research_notes": [],
    "research_rounds": 0,
    "summary": "",
})

print("\n" + "="*60)
print(result["summary"])

## 💡 Key Takeaways

1. **State machines** — LangGraph models workflows as graphs with typed state
2. **Nodes are functions** — Each takes state, returns updated state
3. **Conditional edges** — The graph can branch based on state (loops, fallbacks)
4. **Composability** — Nodes can be swapped, added, or rearranged without rewriting logic

---

<div style="background-color:#507dcb; color:white; padding:10px; border-radius:5px; margin-bottom:5px;"><strong>→ Next:</strong> 03-Graduated-Autonomy.ipynb — Add human-in-the-loop checkpoints where a human can approve, reject, or redirect the agent's work</div>